In [ ]:
%pip install transformers datasets torch torchvision torchaudio seqeval scikit-learn evaluate accelerate

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 1.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.1 MB/s eta 0:00:00
  Created wheel for seqeval: filename=seqeval-1.2.2-py3-none-any.whl size=16162 sha256=c40e714b55474826ca61a4072127187a10bfd727f8f6a1f7de8fa550de5095c3
  Stored in directory: /root/.cache/pip/wheels/5f/b8/73/0b2c1a76b701a677653dd79ece07cfabd7457989dbfbdcd8d7
Successfully built seqeval


In [ ]:
import torch
import numpy as np
from transformers import (
    AutoTokenizer,
    AutoModel,
    AutoConfig,
    TrainingArguments,
    Trainer
)
import pandas as pd
from datasets import Dataset
from torch import nn
from sklearn.metrics import accuracy_score, f1_score, classification_report
from seqeval.metrics import classification_report as seqeval_report, f1_score as seqeval_f1, accuracy_score as seqeval_accuracy
import warnings
warnings.filterwarnings('ignore')

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cpu


In [ ]:
train_df = pd.read_parquet(f"https://huggingface.co/datasets/AmazonScience/massive/resolve/bdbbb020e3ac75ee686f5187495995efb8d5a3b4/pt-PT/massive-train.parquet")
val_df   = pd.read_parquet(f"https://huggingface.co/datasets/AmazonScience/massive/resolve/bdbbb020e3ac75ee686f5187495995efb8d5a3b4/pt-PT/massive-validation.parquet")
test_df  = pd.read_parquet(f"https://huggingface.co/datasets/AmazonScience/massive/resolve/bdbbb020e3ac75ee686f5187495995efb8d5a3b4/pt-PT/massive-test.parquet")

In [ ]:
print(f"Train size: {len(train_df)}")
print(f"Val size: {len(val_df)}")
print(f"Test size: {len(test_df)}")

dataset = {
    'train': Dataset.from_pandas(train_df),
    'validation': Dataset.from_pandas(val_df),
    'test': Dataset.from_pandas(test_df)
}

idx_to_intent = sorted(list(set(train_df['intent'])))
intent_to_idx = {intent: idx for idx, intent in enumerate(idx_to_intent)}
print(f"\nNumber of intents: {len(idx_to_intent)}")
print(f"Sample: {idx_to_intent[:5]}")

example = train_df.iloc[0]
print(f"\nExample utterance: {example['utt']}")
print(f"Intent: {example['intent']}")
print(f"Annotated utterance: {example['annot_utt']}")

Train size: 11514
Val size: 2033
Test size: 2974

Number of intents: 60
Sample: [0, 1, 2, 3, 4]

Example utterance: acorda-me às nove da manhã na sexta-feira
Intent: 48
Annotated utterance: acorda-me às [time : nove da manhã] na [date : sexta-feira]


In [ ]:
def parse_annot_utt_to_bio(annot_utt, utt):

    words = utt.split()
    bio_tags = ['O'] * len(words)

    annot_utt = annot_utt.replace('[', ' [ ').replace(']', ' ] ')

    tokens = annot_utt.split()
    utt_idx = 0
    i = 0

    while i < len(tokens):
        if tokens[i] == '[':
            slot_tokens = []
            i += 1

            while i < len(tokens) and tokens[i] != ':':
                slot_tokens.append(tokens[i])
                i += 1

            slot_name = '_'.join(slot_tokens)
            i += 1

            value_tokens = []
            while i < len(tokens) and tokens[i] != ']':
                value_tokens.append(tokens[i])
                i += 1

            for j in range(len(value_tokens)):
                if utt_idx < len(words):
                    if j == 0:
                        bio_tags[utt_idx] = f'B-{slot_name}'
                    else:
                        bio_tags[utt_idx] = f'I-{slot_name}'
                    utt_idx += 1

            i += 1
        else:
            if utt_idx < len(words):
                utt_idx += 1
            i += 1

    return bio_tags

test_example = dataset['train'][10]
bio_tags = parse_annot_utt_to_bio(test_example['annot_utt'], test_example['utt'])
print(f"utterance: {test_example['utt']}")
print(f"annotated: {test_example['annot_utt']}")
print(f"words: {test_example['utt'].split()}")
print(f"bio tags: {bio_tags}")

utterance: desligar a luz da casa de banho
annotated: desligar a luz da [house_place : casa de banho]
words: ['desligar', 'a', 'luz', 'da', 'casa', 'de', 'banho']
bio tags: ['O', 'O', 'O', 'O', 'B-house_place', 'I-house_place', 'I-house_place']


In [ ]:
def get_all_slot_names(dataset_split):
    slot_names = set()
    for example in dataset_split:
        tokens = example['annot_utt'].split()
        i = 0
        while i < len(tokens):
            if tokens[i] == '[':
                slot_tokens = []
                i += 1
                while i < len(tokens) and tokens[i] != ':':
                    slot_tokens.append(tokens[i])
                    i += 1
                slot_name = '_'.join(slot_tokens)
                slot_names.add(slot_name)
            i += 1
    return slot_names

all_slots = set()

for df in [train_df, val_df, test_df]:
    for _, row in df.iterrows():
        annot_utt = row['annot_utt'].replace('[', ' [ ').replace(']', ' ] ')
        tokens = annot_utt.split()
        i = 0
        while i < len(tokens):
            if tokens[i] == '[':
                slot_tokens = []
                i += 1
                while i < len(tokens) and tokens[i] != ':':
                    slot_tokens.append(tokens[i])
                    i += 1
                slot_name = '_'.join(slot_tokens)
                all_slots.add(slot_name)
            i += 1

print(f"num of unique: {len(all_slots)}")
print(f"sample: {sorted(list(all_slots))[:10]}")

num of unique: 55
sample: ['alarm_type', 'app_name', 'artist_name', 'audiobook_author', 'audiobook_name', 'business_name', 'business_type', 'change_amount', 'coffee_type', 'color_type']


In [ ]:
bio_tags_list = ['O']
for slot in sorted(all_slots):
    bio_tags_list.append(f'B-{slot}')
    bio_tags_list.append(f'I-{slot}')

tag2idx = {tag: idx for idx, tag in enumerate(bio_tags_list)}
idx2tag = {idx: tag for tag, idx in tag2idx.items()}

print(f"\ntotal BIO tags: {len(bio_tags_list)}")
print(f"sample BIO tags: {bio_tags_list[:10]}")


total BIO tags: 111
sample BIO tags: ['O', 'B-alarm_type', 'I-alarm_type', 'B-app_name', 'I-app_name', 'B-artist_name', 'I-artist_name', 'B-audiobook_author', 'I-audiobook_author', 'B-audiobook_name']


In [ ]:
MODEL_NAME = "neuralmind/bert-base-portuguese-cased"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
print(f"Loaded tokenizer: {MODEL_NAME}")
print(f"Vocab size: {len(tokenizer)}")

tokenizer_config.json:   0%|          | 0.00/43.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/647 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

Loaded tokenizer: neuralmind/bert-base-portuguese-cased
Vocab size: 29794


In [ ]:
from datasets import DatasetDict

dataset = DatasetDict({
    'train': Dataset.from_pandas(train_df),
    'validation': Dataset.from_pandas(val_df),
    'test': Dataset.from_pandas(test_df)
})

def tokenize_and_align_labels(examples, max_length=128):
    bio_labels = []
    for utt, annot_utt in zip(examples['utt'], examples['annot_utt']):
        bio_tags = parse_annot_utt_to_bio(annot_utt, utt)
        bio_labels.append(bio_tags)

    words = [utt.split() for utt in examples['utt']]
    tokenized_inputs = tokenizer(
        words,
        truncation=True,
        padding='max_length',
        max_length=max_length,
        is_split_into_words=True
    )

    aligned_labels = []
    for i, label in enumerate(bio_labels):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        aligned_label = []
        previous_word_idx = None

        for word_idx in word_ids:
            if word_idx is None:
                aligned_label.append(-100)
            elif word_idx != previous_word_idx:
                if word_idx < len(label):
                    aligned_label.append(tag2idx[label[word_idx]])
                else:
                    aligned_label.append(tag2idx['O'])
            else:
                if word_idx < len(label):
                    current_tag = label[word_idx]
                    if current_tag.startswith('B-'):
                        current_tag = 'I-' + current_tag[2:]
                    aligned_label.append(tag2idx[current_tag])
                else:
                    aligned_label.append(tag2idx['O'])

            previous_word_idx = word_idx

        aligned_labels.append(aligned_label)

    tokenized_inputs['labels_ner'] = aligned_labels
    tokenized_inputs['labels_intent'] = [intent_to_idx[intent] for intent in examples['intent']]

    return tokenized_inputs

tokenized_datasets = dataset.map(
    tokenize_and_align_labels,
    batched=True,
    remove_columns=dataset['train'].column_names
)

print(f"Sample tokenized example: {tokenized_datasets['train'][0].keys()}")

Map:   0%|          | 0/11514 [00:00<?, ? examples/s]

Map:   0%|          | 0/2033 [00:00<?, ? examples/s]

Map:   0%|          | 0/2974 [00:00<?, ? examples/s]

Tokenization complete!
Sample tokenized example: dict_keys(['input_ids', 'token_type_ids', 'attention_mask', 'labels_ner', 'labels_intent'])


In [ ]:
class MultiTaskModel(nn.Module):
    def __init__(self, model_name, num_intents, num_ner_tags):
        super(MultiTaskModel, self).__init__()

        self.encoder = AutoModel.from_pretrained(model_name)
        hidden_size = self.encoder.config.hidden_size

        # intent
        self.intent_classifier = nn.Sequential(
            nn.Dropout(0.1),
            nn.Linear(hidden_size, num_intents)
        )

        # NER
        self.ner_classifier = nn.Sequential(
            nn.Dropout(0.1),
            nn.Linear(hidden_size, num_ner_tags)
        )

    def forward(self, input_ids, attention_mask, labels_intent=None, labels_ner=None):
        outputs = self.encoder(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        sequence_output = outputs.last_hidden_state

        cls_output = sequence_output[:, 0, :]
        intent_logits = self.intent_classifier(cls_output)

        ner_logits = self.ner_classifier(sequence_output)

        total_loss = None
        if labels_intent is not None and labels_ner is not None:
            loss_fct_intent = nn.CrossEntropyLoss()
            intent_loss = loss_fct_intent(intent_logits, labels_intent)

            loss_fct_ner = nn.CrossEntropyLoss(ignore_index=-100)
            ner_loss = loss_fct_ner(ner_logits.view(-1, ner_logits.size(-1)), labels_ner.view(-1))

            total_loss = intent_loss + ner_loss

        return {
            'loss': total_loss,
            'intent_logits': intent_logits,
            'ner_logits': ner_logits
        }

model = MultiTaskModel(
    model_name=MODEL_NAME,
    num_intents=len(idx_to_intent),
    num_ner_tags=len(bio_tags_list)
)

model.to(device)
print(f"Model initialized with {sum(p.numel() for p in model.parameters()):,} parameters")

pytorch_model.bin:   0%|          | 0.00/438M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/438M [00:00<?, ?B/s]

Model initialized with 109,054,635 parameters


In [ ]:
ner_label_counts = {}
for example in val_dataset:
    for label in example['labels_ner']:
        if label != -100:
            tag = idx2tag[label]
            ner_label_counts[tag] = ner_label_counts.get(tag, 0) + 1

print(f"Total NER tags in validation: {sum(ner_label_counts.values())}")
print(f"Unique NER tags: {len(ner_label_counts)}")
print(f"Top 10 tags: {sorted(ner_label_counts.items(), key=lambda x: x[1], reverse=True)[:10]}")
non_o_count = sum(count for tag, count in ner_label_counts.items() if tag != 'O')
print(f"Non-O tags: {non_o_count} ({non_o_count/sum(ner_label_counts.values())*100:.1f}%)")


Total NER tags in validation: 17761
Unique NER tags: 106
Top 10 tags: [('O', 12854), ('I-date', 437), ('B-date', 337), ('I-person', 282), ('I-place_name', 259), ('I-time', 229), ('B-place_name', 210), ('I-media_type', 166), ('B-event_name', 160), ('B-time', 133)]
Non-O tags: 4907 (27.6%)


In [ ]:
def compute_metrics(pred):

    intent_logits = pred.predictions[0]
    ner_logits = pred.predictions[1]
    intent_labels = pred.label_ids[0]
    ner_labels = pred.label_ids[1]

    intent_preds = np.argmax(intent_logits, axis=1)

    intent_accuracy = accuracy_score(intent_labels, intent_preds)
    intent_f1 = f1_score(intent_labels, intent_preds, average='weighted', zero_division=0)

    ner_preds = np.argmax(ner_logits, axis=2)

    true_ner_labels = []
    pred_ner_labels = []

    for i in range(len(ner_labels)):
        true_labels = []
        pred_labels = []
        for j in range(len(ner_labels[i])):
            if ner_labels[i][j] != -100:
                true_labels.append(idx2tag[int(ner_labels[i][j])])
                pred_labels.append(idx2tag[int(ner_preds[i][j])])

        if true_labels:
            true_ner_labels.append(true_labels)
            pred_ner_labels.append(pred_labels)

    if true_ner_labels and any(label != 'O' for seq in true_ner_labels for label in seq):
        ner_f1 = seqeval_f1(true_ner_labels, pred_ner_labels, zero_division=0)
        ner_accuracy = seqeval_accuracy(true_ner_labels, pred_ner_labels)
    else:
        ner_f1 = 0.0
        ner_accuracy = 0.0

    e2e_correct = 0
    valid_count = len(true_ner_labels)

    for i in range(min(len(intent_preds), len(true_ner_labels))):
        intent_match = (intent_preds[i] == intent_labels[i])
        ner_match = all(t == p for t, p in zip(true_ner_labels[i], pred_ner_labels[i]))
        if intent_match and ner_match:
            e2e_correct += 1

    e2e_accuracy = e2e_correct / valid_count if valid_count > 0 else 0.0

    return {
        'intent_accuracy': intent_accuracy,
        'intent_f1': intent_f1,
        'ner_f1': ner_f1,
        'ner_accuracy': ner_accuracy,
        'e2e_accuracy': e2e_accuracy
    }

In [ ]:
class MultiTaskTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        labels_intent = inputs.pop("labels_intent")
        labels_ner = inputs.pop("labels_ner")

        outputs = model(
            input_ids=inputs["input_ids"],
            attention_mask=inputs["attention_mask"],
            labels_intent=labels_intent,
            labels_ner=labels_ner
        )

        loss = outputs['loss']
        return (loss, outputs) if return_outputs else loss

    def prediction_step(self, model, inputs, prediction_loss_only, ignore_keys=None):
        labels_intent = inputs.pop("labels_intent")
        labels_ner = inputs.pop("labels_ner")

        with torch.no_grad():
            outputs = model(
                input_ids=inputs["input_ids"],
                attention_mask=inputs["attention_mask"],
                labels_intent=labels_intent,
                labels_ner=labels_ner
            )
            loss = outputs['loss']
            intent_logits = outputs['intent_logits']
            ner_logits = outputs['ner_logits']

        if prediction_loss_only:
            return (loss, None, None)

        return (
            loss,
            (intent_logits.detach(), ner_logits.detach()),
            (labels_intent.detach(), labels_ner.detach())
        )

training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    warmup_steps=500,
    weight_decay=0.01,
    logging_dir='./logs',
    logging_steps=100,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="e2e_accuracy",
    greater_is_better=True,
    report_to="none"
)

trainer = MultiTaskTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics
)

print(f"Training for {training_args.num_train_epochs} epochs")
print(f"Batch size: {training_args.per_device_train_batch_size}")

Training for 3 epochs
Batch size: 16


In [ ]:
train_result = trainer.train()

print("\nTraining completed!")
print(f"Train loss: {train_result.training_loss:.4f}")

Epoch,Training Loss,Validation Loss


In [ ]:
test_results = trainer.evaluate(test_dataset)

print(f"Intent Accuracy: {test_results['eval_intent_accuracy']:.4f}")
print(f"Intent F1: {test_results['eval_intent_f1']:.4f}")
print(f"NER F1 (seqeval): {test_results['eval_ner_f1']:.4f}")
print(f"NER Accuracy (seqeval): {test_results['eval_ner_accuracy']:.4f}")
print(f"End-to-End Accuracy: {test_results['eval_e2e_accuracy']:.4f}")

In [ ]:
def get_detailed_predictions(dataset_split, split_name="test"):
    predictions = trainer.predict(dataset_split)

    intent_logits = predictions.predictions[0]
    ner_logits = predictions.predictions[1]
    intent_labels = predictions.label_ids[0]
    ner_labels = predictions.label_ids[1]

    intent_preds = np.argmax(intent_logits, axis=1)
    ner_preds = np.argmax(ner_logits, axis=2)

    true_ner_labels = []
    pred_ner_labels = []

    for i in range(len(ner_labels)):
        true_labels = []
        pred_labels = []
        for j in range(len(ner_labels[i])):
            if ner_labels[i][j] != -100:
                true_labels.append(idx2tag[ner_labels[i][j]])
                pred_labels.append(idx2tag[ner_preds[i][j]])
        true_ner_labels.append(true_labels)
        pred_ner_labels.append(pred_labels)

    print(f"\nDetailed NER Report ({split_name} set):")
    print(seqeval_report(true_ner_labels, pred_ner_labels))

    from collections import Counter
    intent_counts = Counter(intent_labels)
    print(f"\nTop 10 most frequent intents in {split_name} set:")
    for intent_idx, count in intent_counts.most_common(10):
        correct = sum((intent_preds == intent_idx) & (intent_labels == intent_idx))
        total = sum(intent_labels == intent_idx)
        accuracy = correct / total if total > 0 else 0
        print(f"{idx_to_intent[intent_idx]}: {accuracy:.3f} ({correct}/{total})")

    return intent_preds, pred_ner_labels

intent_preds, ner_preds = get_detailed_predictions(test_dataset, "test")

In [ ]:
def process_user_query(text, model, tokenizer):
    words = text.split()
    inputs = tokenizer(
        words,
        truncation=True,
        padding=True,
        max_length=128,
        is_split_into_words=True,
        return_tensors="pt"
    )

    word_ids = inputs.word_ids(batch_index=0)

    inputs = {k: v.to(device) for k, v in inputs.items()}

    model.eval()
    with torch.no_grad():
        outputs = model(
            input_ids=inputs['input_ids'],
            attention_mask=inputs['attention_mask']
        )

    intent_logits = outputs['intent_logits']
    intent_idx = torch.argmax(intent_logits, dim=1).item()
    intent_name = idx_to_intent[intent_idx]

    ner_logits = outputs['ner_logits']
    ner_preds = torch.argmax(ner_logits, dim=2)[0].cpu().numpy()

    word_tags = {}

    for idx, word_idx in enumerate(word_ids):
        if word_idx is not None and word_idx < len(words):
            tag = idx2tag[ner_preds[idx]]
            if word_idx not in word_tags:
                word_tags[word_idx] = tag

    slots = []
    current_slot = None
    current_value = []

    for word_idx in sorted(word_tags.keys()):
        tag = word_tags[word_idx]
        word = words[word_idx]

        if tag.startswith('B-'):
            if current_slot and current_value:
                slots.append({current_slot: ' '.join(current_value)})
            current_slot = tag[2:]
            current_value = [word]
        elif tag.startswith('I-'):
            if current_slot == tag[2:]:
                current_value.append(word)
            else:
                if current_slot and current_value:
                    slots.append({current_slot: ' '.join(current_value)})
                current_slot = tag[2:]
                current_value = [word]
        else:
            if current_slot and current_value:
                slots.append({current_slot: ' '.join(current_value)})
            current_slot = None
            current_value = []

    if current_slot and current_value:
        slots.append({current_slot: ' '.join(current_value)})

    return {
        'text': text,
        'intent': intent_name,
        'slots': slots
    }


In [ ]:
# Примеры пользовательских запросов на португальском
test_queries = [
    "qual é o tempo em Lisboa hoje",
    "defina um alarme para as sete da manhã",
    "toque música de rock",
    "quanto é quinze mais vinte e três",
    "reserve um restaurante italiano para amanhã às oito"
]

for query in test_queries:
    result = process_user_query(query, model, tokenizer)
    print(f"Query: {result['text']}")
    print(f"Intent: {result['intent']}")
    print(f"Slots: {result['slots']}")

In [ ]:
model_save_path = "./portuguese_nlu_model"
model.encoder.save_pretrained(model_save_path)
tokenizer.save_pretrained(model_save_path)

import json

model_config = {
    'base_model': MODEL_NAME,
    'num_intents': len(idx_to_intent),
    'num_ner_tags': len(bio_tags_list),
    'idx_to_intent': idx_to_intent,
    'idx2tag': idx2tag,
    'tag2idx': tag2idx
}

with open(f"{model_save_path}/model_config.json", 'w', encoding='utf-8') as f:
    json.dump(model_config, f, ensure_ascii=False, indent=2)

torch.save({
    'intent_classifier': model.intent_classifier.state_dict(),
    'ner_classifier': model.ner_classifier.state_dict()
}, f"{model_save_path}/classifier_heads.pt")

print(f"Model saved to {model_save_path}")

In [ ]:
print(f"Базовая модель: {MODEL_NAME}")
print(f"Количество интентов: {len(idx_to_intent)}")
print(f"Количество BIO тегов: {len(bio_tags_list)}")
print(f"Язык: Португальский (pt-PT)")
print(f"Модель: {MODEL_NAME}")
print(f"Intent Accuracy: {test_results['eval_intent_accuracy']:.4f}")
print(f"Intent F1: {test_results['eval_intent_f1']:.4f}")
print(f"NER F1: {test_results['eval_ner_f1']:.4f}")
print(f"NER Accuracy: {test_results['eval_ner_accuracy']:.4f}")
print(f"End-to-End Accuracy: {test_results['eval_e2e_accuracy']:.4f}")